# Stage 3 — Data Cleaning and Preprocessing

This stage standardizes column names, detects year columns, removes non-country aggregates, identifies the U5MR target indicator, and prepares clean country-level WDI data.

Feature selection is not performed in this stage.

## 3.1 Standardize column names and numeric year values

In [294]:
wdi = raw.rename(columns={
    "REF_AREA": "country_code",
    "REF_AREA_LABEL": "country_name",
    "INDICATOR": "indicator_code",
    "INDICATOR_LABEL": "indicator_name"
}).copy()

year_columns = sorted([c for c in wdi.columns if re.fullmatch(r"\d{4}", str(c))], key=int)

for col in year_columns:
    wdi[col] = pd.to_numeric(wdi[col], errors="coerce")

print("Detected year range:", year_columns[0], "to", year_columns[-1])
print("Number of year columns:", len(year_columns))

Detected year range: 1960 to 2025
Number of year columns: 66


## 3.1.1 Verify data types after cleaning

After renaming columns, all year columns are converted to numeric values. This confirms that the ML features will be numeric and that metadata remains categorical/text.

In [296]:
post_conversion_dtype_audit = pd.DataFrame({
    "column": wdi.columns,
    "dtype_after_cleaning": wdi.dtypes.astype(str).values,
    "is_year_column": [str(col) in year_columns for col in wdi.columns],
    "non_null_count": wdi.notna().sum().values,
    "missing_count": wdi.isna().sum().values,
    "missing_pct": wdi.isna().mean().values
})

year_type_counts_after = (
    post_conversion_dtype_audit[post_conversion_dtype_audit["is_year_column"]]
    ["dtype_after_cleaning"]
    .value_counts()
    .rename_axis("dtype_after_cleaning")
    .reset_index(name="number_of_year_columns")
)

metadata_types_after = post_conversion_dtype_audit[
    ~post_conversion_dtype_audit["is_year_column"]
][["column", "dtype_after_cleaning", "missing_pct"]]

display(year_type_counts_after)
display(metadata_types_after)

post_conversion_dtype_audit.to_csv(TABLE_DIR / "post_cleaning_dtype_audit.csv", index=False)

,dtype_after_cleaning,number_of_year_columns
0,float64,66


,column,dtype_after_cleaning,missing_pct
0,STRUCTURE,object,0.0
1,STRUCTURE_ID,object,0.0
2,ACTION,object,0.0
3,FREQ,object,0.0
4,country_code,object,0.0
5,indicator_code,object,0.0
6,SEX,object,0.0
7,AGE,object,0.0
8,URBANISATION,object,0.0
9,UNIT_MEASURE,object,0.0


## 3.2 Remove regional and income-group aggregates

In [298]:
def get_iso3_country_codes():
    '''Return ISO3 country/economy codes. Kosovo (XKX) is added manually because it appears in many World Bank files.'''
    if pycountry is None:
        return None
    codes = {country.alpha_3 for country in pycountry.countries}
    codes.add("XKX")
    return codes

iso3_codes = get_iso3_country_codes()

if iso3_codes is not None:
    country_wdi = wdi[wdi["country_code"].isin(iso3_codes)].copy()
else:
    country_wdi = wdi[wdi["country_code"].astype(str).str.fullmatch(r"[A-Z]{3}", na=False)].copy()

before_duplicates = len(country_wdi)
country_wdi = country_wdi.drop_duplicates(subset=["country_code", "indicator_code"], keep="first")
removed_duplicates = before_duplicates - len(country_wdi)

print("Rows after country filter:", country_wdi.shape[0])
print("Countries/economies after country filter:", country_wdi["country_code"].nunique())
print("Indicators after country filter:", country_wdi["indicator_code"].nunique())
print("Duplicate country-indicator rows removed:", removed_duplicates)

Rows after country filter: 183101
Countries/economies after country filter: 216
Indicators after country filter: 1514
Duplicate country-indicator rows removed: 0


## 3.3 Build the indicator inventory

In [300]:
indicator_inventory = (
    country_wdi[["indicator_code", "indicator_name"]]
    .drop_duplicates()
    .sort_values("indicator_code")
    .reset_index(drop=True)
)

indicator_inventory.to_csv(TABLE_DIR / "indicator_inventory.csv", index=False)
print("Indicator inventory size:", len(indicator_inventory))
display(indicator_inventory.head(10))

Indicator inventory size: 1514


,indicator_code,indicator_name
0,WB_WDI_AG_CON_FERT_PT_ZS,Fertilizer consumption (% of fertilizer produc...
1,WB_WDI_AG_CON_FERT_ZS,Fertilizer consumption (kilograms per hectare ...
2,WB_WDI_AG_LND_AGRI_K2,Agricultural land (sq. km)
3,WB_WDI_AG_LND_AGRI_ZS,Agricultural land (% of land area)
4,WB_WDI_AG_LND_ARBL_HA,Arable land (hectares)
5,WB_WDI_AG_LND_ARBL_HA_PC,Arable land (hectares per person)
6,WB_WDI_AG_LND_ARBL_ZS,Arable land (% of land area)
7,WB_WDI_AG_LND_CREL_HA,Land under cereal production (hectares)
8,WB_WDI_AG_LND_CROP_ZS,Permanent cropland (% of land area)
9,WB_WDI_AG_LND_EL5M_RU_K2,Rural land area where elevation is below 5 met...


## 3.4 Identify the U5MR target indicator

In [302]:
u5mr_candidates = indicator_inventory[
    indicator_inventory["indicator_code"].astype(str).str.contains("SH_DYN_MORT|SH.DYN.MORT", case=False, regex=True, na=False)
    | indicator_inventory["indicator_name"].astype(str).str.contains("under-5|under five|under-five", case=False, regex=True, na=False)
].copy()

display(u5mr_candidates)

TARGET_INDICATOR_CODE = "WB_WDI_SH_DYN_MORT"

if TARGET_INDICATOR_CODE not in set(indicator_inventory["indicator_code"]):
    raise ValueError("Expected U5MR indicator WB_WDI_SH_DYN_MORT was not found. Inspect u5mr_candidates above.")

u5mr_wide = country_wdi[country_wdi["indicator_code"] == TARGET_INDICATOR_CODE].copy()
print("U5MR rows:", len(u5mr_wide))

,indicator_code,indicator_name
1002,WB_WDI_SH_DTH_MORT,Number of under-five deaths
1011,WB_WDI_SH_DYN_MORT,"Mortality rate, under-5 (per 1,000 live births)"
1012,WB_WDI_SH_DYN_MORT_FE,"Mortality rate, under-5, female (per 1,000 liv..."
1013,WB_WDI_SH_DYN_MORT_MA,"Mortality rate, under-5, male (per 1,000 live ..."
1046,WB_WDI_SH_MLR_NETS_ZS,Use of insecticide-treated bed nets (% of unde...


U5MR rows: 146


## 3.5 Check U5MR availability by year

In [304]:
u5mr_availability = pd.DataFrame({
    "year": [int(c) for c in year_columns],
    "available_countries": [u5mr_wide[c].notna().sum() for c in year_columns]
})

u5mr_availability.to_csv(TABLE_DIR / "u5mr_availability_by_year.csv", index=False)
display(u5mr_availability.tail(12))

,year,available_countries
54,2014,146
55,2015,146
56,2016,146
57,2017,146
58,2018,146
59,2019,146
60,2020,146
61,2021,146
62,2022,146
63,2023,146


## 3.6 Select the target year and modern working period

In [306]:
if str(TARGET_YEAR) not in year_columns:
    raise ValueError(f"Target year {TARGET_YEAR} is not available as a column.")

if str(PREDICTOR_LEVEL_YEAR) not in year_columns:
    raise ValueError(f"Predictor year {PREDICTOR_LEVEL_YEAR} is not available as a column.")

if str(CHANGE_START_YEAR) not in year_columns or str(CHANGE_END_YEAR) not in year_columns:
    raise ValueError("The requested change-window years are not available as columns.")

if u5mr_wide[str(TARGET_YEAR)].notna().sum() < 100:
    raise ValueError("U5MR 2023 has too few observed values for the planned cross-sectional model.")

assert PREDICTOR_LEVEL_YEAR < TARGET_YEAR, "Predictor level year must be before the target year."
assert CHANGE_END_YEAR < TARGET_YEAR, "Change window must end before the target year."

print("Target year:", TARGET_YEAR)
print("Predictor level year:", PREDICTOR_LEVEL_YEAR)
print("Historical audit years:", PREDICTOR_YEARS[0], "to", PREDICTOR_YEARS[-1])
print("Change feature window:", CHANGE_START_YEAR, "to", CHANGE_END_YEAR)
print("U5MR countries available in target year:", u5mr_wide[str(TARGET_YEAR)].notna().sum())

Target year: 2023
Predictor level year: 2022
Historical audit years: 2000 to 2022
Change feature window: 2018 to 2022
U5MR countries available in target year: 146


## 3.7 Create the target table

In [308]:
target_df = (
    u5mr_wide[["country_code", "country_name", str(TARGET_YEAR)]]
    .rename(columns={str(TARGET_YEAR): "u5mr_2023"})
    .dropna(subset=["u5mr_2023"])
    .reset_index(drop=True)
)

target_df["log_u5mr_2023"] = np.log(target_df["u5mr_2023"])

target_df.to_csv(DATA_DIR / "target_u5mr_2023.csv", index=False)
print("Countries with observed U5MR 2023:", len(target_df))
display(target_df.head())

Countries with observed U5MR 2023: 146


,country_code,country_name,u5mr_2023,log_u5mr_2023
0,NAM,Namibia,40.7,3.706228
1,NER,Niger,114.8,4.743191
2,SSD,South Sudan,98.7,4.592085
3,MEX,Mexico,12.5,2.525729
4,MUS,Mauritius,15.2,2.721295


### Target measurement note

The target used for supervised learning is the WDI-reported value of under-five mortality in 2023. In an applied public-health setting, recent U5MR series may include statistical estimation and smoothing rather than direct annual vital-registration counts for every country. Therefore, the target is treated as the best available reported outcome for this project, while acknowledging that recent U5MR values may contain measurement uncertainty. This does not change the regression task, but it matters for cautious interpretation of model performance.